In [ ]:
# ============================================================
# Deep Learning rolling-origin benchmark. ready for google colab
#
# Models:
#   1) DL_GlobalGRU_all_infections_all_countries
#   2) DL_GlobalTransformer_all_infections_all_countries
#   3) DL_DiseaseTransformer_all_countries
#   4) DL_LocalTCN_each_country
#
# Evaluation:
#   - Train: 2014-2022
#   - Test : 2023
#   - Horizons: 1,2,3,4 weeks ahead
#   - Fixed countries to match local notebooks:
#       ARI: BE, BG, CZ, DE, EE, LT, RO, SI
#       ILI: BE, CZ, EE, GR, IE, LT, NL, PL, RO, SI
# ============================================================

import os
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)

from tensorflow.keras import layers, models, callbacks, optimizers, losses
from google.colab import files
from IPython.display import display

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


# ============================================================
# 1. Global settings
# ============================================================

M_SEASON = 52
HORIZONS = (1, 2, 3, 4)
MAX_H = max(HORIZONS)

SEQ_LEN = 104

TRAIN_START = pd.Timestamp("2014-01-05")
TRAIN_END = pd.Timestamp("2022-12-25")
TEST_START = pd.Timestamp("2023-01-01")
TEST_END = pd.Timestamp("2023-12-31")

BATCH_SIZE = 64

EPOCHS_GLOBAL = 45
EPOCHS_DISEASE = 45
EPOCHS_LOCAL = 30

PATIENCE_GLOBAL = 6
PATIENCE_DISEASE = 6
PATIENCE_LOCAL = 4

CLIP_NEGATIVE_PREDICTIONS = False

OUT_DIR = Path("/content/dl_outputs_fixed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_GLOBAL_GRU = "DL_GlobalGRU_all_infections_all_countries"
MODEL_GLOBAL_TRANSFORMER = "DL_GlobalTransformer_all_infections_all_countries"
MODEL_DISEASE_TRANSFORMER = "DL_DiseaseTransformer_all_countries"
MODEL_LOCAL_TCN = "DL_LocalTCN_each_country"

# Fixed country coverage, matching local notebooks
OK_ARI_FIXED = ["BE", "BG", "CZ", "DE", "EE", "LT", "RO", "SI"]
OK_ILI_FIXED = ["BE", "CZ", "EE", "GR", "IE", "LT", "NL", "PL", "RO", "SI"]


# ============================================================
# 2. Upload and load data
# ============================================================

print("\nUpload the two raw incidence CSV files: one ARI and one ILI.")
uploaded = files.upload()
uploaded_names = list(uploaded.keys())

print("\nUploaded files:")
print(uploaded_names)


def find_uploaded_file_for_disease(disease_name):
    names_lower = {name: name.lower() for name in uploaded_names}

    if disease_name == "ARI":
        candidates = [
            name for name, low in names_lower.items()
            if "ari" in low and "ili" not in low
        ]
    elif disease_name == "ILI":
        candidates = [
            name for name, low in names_lower.items()
            if "ili" in low
        ]
    else:
        raise ValueError("Disease must be ARI or ILI.")

    if len(candidates) == 0:
        raise FileNotFoundError(f"No uploaded file detected for {disease_name}.")

    if len(candidates) > 1:
        print(f"Multiple candidates for {disease_name}. Using:", candidates[0])

    return candidates[0]


def read_disease_csv(path):
    df = pd.read_csv(path)

    possible_location_cols = ["location", "country", "geo_value", "geo_id"]
    possible_date_cols = ["truth_date", "date", "week", "time_value", "target_end_date"]
    possible_value_cols = ["value", "incidence", "rate", "target", "y"]

    location_col = next((c for c in possible_location_cols if c in df.columns), None)
    date_col = next((c for c in possible_date_cols if c in df.columns), None)
    value_col = next((c for c in possible_value_cols if c in df.columns), None)

    if location_col is None or date_col is None or value_col is None:
        raise ValueError(
            f"Could not infer required columns from {path}. "
            f"Columns found: {list(df.columns)}"
        )

    out = df[[location_col, date_col, value_col]].copy()
    out.columns = ["location", "truth_date", "value"]

    out["truth_date"] = pd.to_datetime(out["truth_date"])
    out["value"] = pd.to_numeric(out["value"], errors="coerce")

    out = out.sort_values(["location", "truth_date"]).reset_index(drop=True)

    return out


ari_file = find_uploaded_file_for_disease("ARI")
ili_file = find_uploaded_file_for_disease("ILI")

ari_df = read_disease_csv(ari_file)
ili_df = read_disease_csv(ili_file)

print("\nARI shape:", ari_df.shape)
print("ILI shape:", ili_df.shape)

display(ari_df.head())
display(ili_df.head())


# ============================================================
# 3. Calendar and helpers
# ============================================================

full_weeks = pd.date_range(
    start=TRAIN_START,
    end=max(ari_df["truth_date"].max(), ili_df["truth_date"].max()),
    freq="W-SUN"
)

train_weeks = full_weeks[
    (full_weeks >= TRAIN_START) &
    (full_weeks <= TRAIN_END)
]

test_weeks = full_weeks[
    (full_weeks >= TEST_START) &
    (full_weeks <= TEST_END)
]

print("\ntrain weeks:", len(train_weeks), train_weeks.min(), "->", train_weeks.max())
print("test weeks :", len(test_weeks), test_weeks.min(), "->", test_weeks.max())


def extract_raw_weekly_series(df, location, calendar):
    sub = df[df["location"] == location].copy()

    if len(sub) == 0:
        return pd.Series(index=calendar, dtype=float)

    s = (
        sub.groupby("truth_date")["value"]
        .mean()
        .sort_index()
        .astype(float)
    )

    return s.reindex(calendar)


def impute_train_series(df, location):
    raw = extract_raw_weekly_series(df, location, train_weeks)

    if raw.notna().sum() == 0:
        return raw

    first_obs = raw.first_valid_index()
    raw = raw.loc[first_obs:].copy()

    s = raw.astype(float)
    s = s.interpolate(method="time", limit_direction="both")
    s = s.ffill().bfill()

    return s.astype(float)


def mase_scale_seasonal(series, m=52):
    s = pd.Series(series).dropna().astype(float)

    if len(s) <= m:
        diff = s.diff().dropna().abs()
    else:
        diff = (s - s.shift(m)).dropna().abs()

    scale = float(diff.mean()) if len(diff) else np.nan

    if not np.isfinite(scale) or scale <= 1e-12:
        fallback = s.diff().dropna().abs()
        scale = float(fallback.mean()) if len(fallback) else 1.0

    if not np.isfinite(scale) or scale <= 1e-12:
        scale = 1.0

    return scale


def mae_np(y, pred):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(pred))))


def rmse_np(y, pred):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(pred)) ** 2)))


def mase_from_mae(m_mae, scale):
    if not np.isfinite(scale) or scale <= 1e-12:
        return np.nan
    return float(m_mae / scale)


def make_week_features(dates):
    weeks = np.array([min(int(d.isocalendar().week), 52) for d in dates], dtype=float)
    sin_week = np.sin(2 * np.pi * weeks / 52.0)
    cos_week = np.cos(2 * np.pi * weeks / 52.0)
    return sin_week, cos_week


def sequence_features(values, dates, mean, std):
    values = pd.Series(values).astype(float)

    if values.isna().any():
        return None

    y_norm = ((values.values - mean) / std).astype("float32")
    sin_week, cos_week = make_week_features(dates)

    x = np.stack(
        [
            y_norm,
            sin_week.astype("float32"),
            cos_week.astype("float32")
        ],
        axis=1
    )

    return x.astype("float32")


def denormalize_prediction(pred_norm, mean, std):
    pred = float(pred_norm * std + mean)

    if CLIP_NEGATIVE_PREDICTIONS:
        pred = max(0.0, pred)

    return pred


# ============================================================
# 4. Fixed country selection
# ============================================================

available_ari = sorted(ari_df["location"].dropna().unique())
available_ili = sorted(ili_df["location"].dropna().unique())

missing_ari = sorted(set(OK_ARI_FIXED) - set(available_ari))
missing_ili = sorted(set(OK_ILI_FIXED) - set(available_ili))

if missing_ari:
    raise ValueError(f"These fixed ARI countries are missing from uploaded file: {missing_ari}")

if missing_ili:
    raise ValueError(f"These fixed ILI countries are missing from uploaded file: {missing_ili}")

ok_ari = OK_ARI_FIXED.copy()
ok_ili = OK_ILI_FIXED.copy()

print("\nUSING FIXED LOCAL-COMPARABLE COVERAGE")
print("ok_ari:", ok_ari, "n =", len(ok_ari))
print("ok_ili:", ok_ili, "n =", len(ok_ili))


# ============================================================
# 5. Build series registry
# ============================================================

DISEASE_TO_ID = {"ARI": 0, "ILI": 1}

all_locations = sorted(set(ok_ari).union(set(ok_ili)))
LOCATION_TO_ID = {loc: i for i, loc in enumerate(all_locations)}

SERIES_DATA = {}

for disease, df_raw, locations in [
    ("ARI", ari_df, ok_ari),
    ("ILI", ili_df, ok_ili)
]:
    for loc in locations:
        train_s = impute_train_series(df_raw, loc)
        raw_full = extract_raw_weekly_series(df_raw, loc, full_weeks)
        test_s = raw_full.reindex(test_weeks)

        mean = float(train_s.mean())
        std = float(train_s.std(ddof=0))

        if not np.isfinite(std) or std <= 1e-8:
            std = 1.0

        scale = mase_scale_seasonal(train_s, m=M_SEASON)

        SERIES_DATA[(disease, loc)] = {
            "disease": disease,
            "location": loc,
            "train": train_s,
            "raw_full": raw_full,
            "test": test_s,
            "mean": mean,
            "std": std,
            "mase_scale": scale,
            "disease_id": DISEASE_TO_ID[disease],
            "location_id": LOCATION_TO_ID[loc]
        }

ALL_KEYS = sorted(SERIES_DATA.keys())

print("\nTotal disease-country series:", len(ALL_KEYS))
print(ALL_KEYS)


# ============================================================
# 6. Supervised samples and eval inputs
# ============================================================

def build_supervised_samples(keys):
    X_seq = []
    Y = []
    disease_ids = []
    location_ids = []

    for key in keys:
        info = SERIES_DATA[key]

        s = info["train"].dropna().astype(float)
        mean = info["mean"]
        std = info["std"]

        if len(s) < SEQ_LEN + MAX_H + 5:
            continue

        for i in range(SEQ_LEN - 1, len(s) - MAX_H):
            seq = s.iloc[i - SEQ_LEN + 1:i + 1]
            seq_dates = seq.index

            x = sequence_features(seq, seq_dates, mean, std)

            if x is None:
                continue

            y_targets = []
            ok = True

            for h in HORIZONS:
                val = s.iloc[i + h]
                if pd.isna(val):
                    ok = False
                    break
                y_targets.append((float(val) - mean) / std)

            if not ok:
                continue

            X_seq.append(x)
            Y.append(y_targets)
            disease_ids.append(info["disease_id"])
            location_ids.append(info["location_id"])

    X_seq = np.asarray(X_seq, dtype="float32")
    Y = np.asarray(Y, dtype="float32")
    disease_ids = np.asarray(disease_ids, dtype="int32").reshape(-1, 1)
    location_ids = np.asarray(location_ids, dtype="int32").reshape(-1, 1)

    if len(X_seq) == 0:
        raise ValueError("No supervised samples were created.")

    return X_seq, Y, disease_ids, location_ids


def build_eval_inputs(keys):
    X_seq = []
    disease_ids = []
    location_ids = []
    valid_meta = []
    invalid_meta = []

    for key in keys:
        info = SERIES_DATA[key]

        train_s = info["train"]
        raw_full = info["raw_full"]
        y_test = raw_full.reindex(test_weeks)

        available = pd.Series(index=full_weeks, dtype=float)
        available.loc[train_s.index] = train_s.values

        mean = info["mean"]
        std = info["std"]

        for origin in test_weeks:
            if pd.notna(y_test.loc[origin]):
                available.loc[origin] = float(y_test.loc[origin])

            seq_dates = pd.date_range(end=origin, periods=SEQ_LEN, freq="W-SUN")
            seq_vals = available.reindex(seq_dates).ffill()

            x = sequence_features(seq_vals, seq_dates, mean, std)

            meta = {
                "key": key,
                "disease": info["disease"],
                "location": info["location"],
                "origin": origin,
                "mean": mean,
                "std": std
            }

            if x is None:
                invalid_meta.append(meta)
            else:
                X_seq.append(x)
                disease_ids.append(info["disease_id"])
                location_ids.append(info["location_id"])
                valid_meta.append(meta)

    X_seq = np.asarray(X_seq, dtype="float32")
    disease_ids = np.asarray(disease_ids, dtype="int32").reshape(-1, 1)
    location_ids = np.asarray(location_ids, dtype="int32").reshape(-1, 1)

    return X_seq, disease_ids, location_ids, valid_meta, invalid_meta


def predictions_to_long(model_name, valid_meta, pred_norm, invalid_meta=None):
    rows = []

    if invalid_meta is None:
        invalid_meta = []

    for meta, pred_vec in zip(valid_meta, pred_norm):
        key = meta["key"]
        info = SERIES_DATA[key]
        y_test = info["raw_full"].reindex(test_weeks)

        origin = meta["origin"]
        mean = meta["mean"]
        std = meta["std"]

        for h in HORIZONS:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            actual = y_test.loc[target]

            try:
                pred = denormalize_prediction(pred_vec[h - 1], mean, std)
            except Exception:
                pred = np.nan

            rows.append({
                "origin": origin,
                "target": target,
                "disease": meta["disease"],
                "location": meta["location"],
                "h": int(h),
                "y": float(actual) if pd.notna(actual) else np.nan,
                "pred": float(pred) if pd.notna(pred) else np.nan,
                "model": model_name
            })

    for meta in invalid_meta:
        key = meta["key"]
        info = SERIES_DATA[key]
        y_test = info["raw_full"].reindex(test_weeks)

        origin = meta["origin"]

        for h in HORIZONS:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            actual = y_test.loc[target]

            rows.append({
                "origin": origin,
                "target": target,
                "disease": meta["disease"],
                "location": meta["location"],
                "h": int(h),
                "y": float(actual) if pd.notna(actual) else np.nan,
                "pred": np.nan,
                "model": model_name
            })

    return pd.DataFrame(rows)


# ============================================================
# 7. Model builders
# ============================================================

def build_global_gru_model(num_diseases, num_locations):
    seq_in = layers.Input(shape=(SEQ_LEN, 3), name="seq")
    disease_in = layers.Input(shape=(1,), dtype="int32", name="disease_id")
    location_in = layers.Input(shape=(1,), dtype="int32", name="location_id")

    x = layers.GRU(64, return_sequences=False)(seq_in)
    x = layers.Dropout(0.20)(x)

    d_emb = layers.Embedding(num_diseases, 4)(disease_in)
    d_emb = layers.Flatten()(d_emb)

    l_emb = layers.Embedding(num_locations, 8)(location_in)
    l_emb = layers.Flatten()(l_emb)

    x = layers.Concatenate()([x, d_emb, l_emb])
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.15)(x)
    x = layers.Dense(32, activation="relu")(x)

    out = layers.Dense(MAX_H, activation="linear")(x)

    model = models.Model(
        inputs=[seq_in, disease_in, location_in],
        outputs=out,
        name=MODEL_GLOBAL_GRU
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss=losses.Huber(delta=1.0),
        metrics=["mae"]
    )

    return model


def build_global_transformer_model(num_diseases, num_locations, d_model=48, num_heads=4):
    seq_in = layers.Input(shape=(SEQ_LEN, 3), name="seq")
    disease_in = layers.Input(shape=(1,), dtype="int32", name="disease_id")
    location_in = layers.Input(shape=(1,), dtype="int32", name="location_id")

    x = layers.Dense(d_model)(seq_in)

    positions = tf.range(start=0, limit=SEQ_LEN, delta=1)
    pos_emb = layers.Embedding(input_dim=SEQ_LEN, output_dim=d_model)(positions)
    x = x + pos_emb

    for _ in range(2):
        attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=0.10
        )(x, x)

        x = layers.LayerNormalization(epsilon=1e-6)(x + attn)

        ff = layers.Dense(96, activation="relu")(x)
        ff = layers.Dropout(0.10)(ff)
        ff = layers.Dense(d_model)(ff)

        x = layers.LayerNormalization(epsilon=1e-6)(x + ff)

    x = layers.GlobalAveragePooling1D()(x)

    d_emb = layers.Embedding(num_diseases, 4)(disease_in)
    d_emb = layers.Flatten()(d_emb)

    l_emb = layers.Embedding(num_locations, 8)(location_in)
    l_emb = layers.Flatten()(l_emb)

    x = layers.Concatenate()([x, d_emb, l_emb])
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.15)(x)

    out = layers.Dense(MAX_H, activation="linear")(x)

    model = models.Model(
        inputs=[seq_in, disease_in, location_in],
        outputs=out,
        name=MODEL_GLOBAL_TRANSFORMER
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=8e-4),
        loss=losses.Huber(delta=1.0),
        metrics=["mae"]
    )

    return model


def build_disease_transformer_model(num_locations, d_model=48, num_heads=4):
    seq_in = layers.Input(shape=(SEQ_LEN, 3), name="seq")
    location_in = layers.Input(shape=(1,), dtype="int32", name="location_id")

    x = layers.Dense(d_model)(seq_in)

    positions = tf.range(start=0, limit=SEQ_LEN, delta=1)
    pos_emb = layers.Embedding(input_dim=SEQ_LEN, output_dim=d_model)(positions)
    x = x + pos_emb

    for _ in range(2):
        attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=0.10
        )(x, x)

        x = layers.LayerNormalization(epsilon=1e-6)(x + attn)

        ff = layers.Dense(96, activation="relu")(x)
        ff = layers.Dropout(0.10)(ff)
        ff = layers.Dense(d_model)(ff)

        x = layers.LayerNormalization(epsilon=1e-6)(x + ff)

    x = layers.GlobalAveragePooling1D()(x)

    l_emb = layers.Embedding(num_locations, 8)(location_in)
    l_emb = layers.Flatten()(l_emb)

    x = layers.Concatenate()([x, l_emb])
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.15)(x)

    out = layers.Dense(MAX_H, activation="linear")(x)

    model = models.Model(
        inputs=[seq_in, location_in],
        outputs=out,
        name=MODEL_DISEASE_TRANSFORMER
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=8e-4),
        loss=losses.Huber(delta=1.0),
        metrics=["mae"]
    )

    return model


def build_local_tcn_model():
    seq_in = layers.Input(shape=(SEQ_LEN, 3), name="seq")

    x = layers.Conv1D(32, kernel_size=3, padding="causal", dilation_rate=1, activation="relu")(seq_in)
    x = layers.Dropout(0.10)(x)

    x = layers.Conv1D(32, kernel_size=3, padding="causal", dilation_rate=2, activation="relu")(x)
    x = layers.Dropout(0.10)(x)

    x = layers.Conv1D(32, kernel_size=3, padding="causal", dilation_rate=4, activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.10)(x)

    out = layers.Dense(MAX_H, activation="linear")(x)

    model = models.Model(
        inputs=seq_in,
        outputs=out,
        name=MODEL_LOCAL_TCN
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss=losses.Huber(delta=1.0),
        metrics=["mae"]
    )

    return model


def fit_model(model, inputs, y, epochs, patience, batch_size=BATCH_SIZE, verbose=1):
    cb = [
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=max(2, patience // 2),
            min_lr=1e-5
        )
    ]

    hist = model.fit(
        inputs,
        y,
        validation_split=0.15,
        epochs=epochs,
        batch_size=batch_size,
        shuffle=True,
        callbacks=cb,
        verbose=verbose
    )

    return hist


# ============================================================
# 8. Evaluation summaries and audit
# ============================================================

def expected_points_for_key(key):
    info = SERIES_DATA[key]
    y_test = info["raw_full"].reindex(test_weeks)

    rows = []

    for h in HORIZONS:
        n_expected = 0

        for origin in test_weeks:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            if pd.notna(y_test.loc[target]):
                n_expected += 1

        rows.append({
            "disease": info["disease"],
            "location": info["location"],
            "h": int(h),
            "expected_actual_points": int(n_expected)
        })

    return pd.DataFrame(rows)


expected_all = pd.concat(
    [expected_points_for_key(key) for key in ALL_KEYS],
    ignore_index=True
)


def build_country_h(long_df):
    rows = []

    for (model_name, disease, location, h), sub in long_df.groupby(["model", "disease", "location", "h"]):
        sub = sub.copy()
        sub = sub[sub["y"].notna()]
        sub = sub[sub["pred"].notna()]

        if len(sub) == 0:
            continue

        info = SERIES_DATA[(disease, location)]
        scale = info["mase_scale"]

        m_mae = mae_np(sub["y"].values, sub["pred"].values)
        m_rmse = rmse_np(sub["y"].values, sub["pred"].values)
        m_mase = mase_from_mae(m_mae, scale)

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "model": model_name,
            "mae": float(m_mae),
            "rmse": float(m_rmse),
            "mase": float(m_mase),
            "n": int(len(sub))
        })

    return (
        pd.DataFrame(rows)
        .sort_values(["model", "disease", "location", "h"])
        .reset_index(drop=True)
    )


def build_n_audit(country_h, expected_all, intended_coverage):
    scored = (
        country_h
        .groupby(["model", "disease", "location", "h"], as_index=False)
        .agg(scored_points=("n", "sum"))
    )

    expected_by_model = expected_all.merge(
        intended_coverage,
        on=["disease", "location"],
        how="inner"
    )

    audit = expected_by_model.merge(
        scored,
        on=["model", "disease", "location", "h"],
        how="left"
    )

    audit["scored_points"] = audit["scored_points"].fillna(0).astype(int)
    audit["missing_due_to_model_or_pred_nan"] = (
        audit["expected_actual_points"] - audit["scored_points"]
    )

    return audit.sort_values(["model", "disease", "location", "h"]).reset_index(drop=True)


def build_horizon_summary(country_h, disease):
    sub = country_h[country_h["disease"] == disease].copy()

    if len(sub) == 0:
        return pd.DataFrame()

    out = (
        sub
        .groupby(["h", "model"], as_index=False)
        .agg(
            mae=("mae", "mean"),
            rmse=("rmse", "mean"),
            mase=("mase", "mean"),
            n_countries=("location", "nunique"),
            n_points=("n", "sum")
        )
        .sort_values(["h", "mase", "mae"])
        .reset_index(drop=True)
    )

    return out


def add_intended_coverage(rows, model_name, keys):
    for disease, loc in keys:
        rows.append({
            "model": model_name,
            "disease": disease,
            "location": loc
        })


# ============================================================
# 9. Run DL experiments
# ============================================================

long_parts = []
training_status_rows = []
intended_coverage_rows = []


# ------------------------------------------------------------
# 9.1 Global GRU: all infections + all countries
# ------------------------------------------------------------

print("\n============================================================")
print("Training model 1:", MODEL_GLOBAL_GRU)
print("============================================================")

tf.keras.backend.clear_session()

X_seq, Y, D_ids, L_ids = build_supervised_samples(ALL_KEYS)

print("Global GRU training samples:", X_seq.shape, Y.shape)

global_gru = build_global_gru_model(
    num_diseases=len(DISEASE_TO_ID),
    num_locations=len(LOCATION_TO_ID)
)

hist = fit_model(
    global_gru,
    [X_seq, D_ids, L_ids],
    Y,
    epochs=EPOCHS_GLOBAL,
    patience=PATIENCE_GLOBAL,
    verbose=1
)

training_status_rows.append({
    "model": MODEL_GLOBAL_GRU,
    "scope": "all_diseases_all_countries",
    "disease": "ALL",
    "location": "ALL",
    "n_train_samples": len(X_seq),
    "epochs_run": len(hist.history["loss"]),
    "final_loss": float(hist.history["loss"][-1]),
    "final_val_loss": float(hist.history["val_loss"][-1]),
    "error": ""
})

X_eval, D_eval, L_eval, valid_meta, invalid_meta = build_eval_inputs(ALL_KEYS)

pred_norm = global_gru.predict(
    [X_eval, D_eval, L_eval],
    batch_size=256,
    verbose=0
)

long_global_gru = predictions_to_long(
    model_name=MODEL_GLOBAL_GRU,
    valid_meta=valid_meta,
    pred_norm=pred_norm,
    invalid_meta=invalid_meta
)

long_parts.append(long_global_gru)
add_intended_coverage(intended_coverage_rows, MODEL_GLOBAL_GRU, ALL_KEYS)

print("Global GRU long rows:", long_global_gru.shape)


# ------------------------------------------------------------
# 9.2 Global Transformer: all infections + all countries
# ------------------------------------------------------------

print("\n============================================================")
print("Training model 2:", MODEL_GLOBAL_TRANSFORMER)
print("============================================================")

tf.keras.backend.clear_session()

global_transformer = build_global_transformer_model(
    num_diseases=len(DISEASE_TO_ID),
    num_locations=len(LOCATION_TO_ID)
)

hist = fit_model(
    global_transformer,
    [X_seq, D_ids, L_ids],
    Y,
    epochs=EPOCHS_GLOBAL,
    patience=PATIENCE_GLOBAL,
    verbose=1
)

training_status_rows.append({
    "model": MODEL_GLOBAL_TRANSFORMER,
    "scope": "all_diseases_all_countries",
    "disease": "ALL",
    "location": "ALL",
    "n_train_samples": len(X_seq),
    "epochs_run": len(hist.history["loss"]),
    "final_loss": float(hist.history["loss"][-1]),
    "final_val_loss": float(hist.history["val_loss"][-1]),
    "error": ""
})

pred_norm = global_transformer.predict(
    [X_eval, D_eval, L_eval],
    batch_size=256,
    verbose=0
)

long_global_transformer = predictions_to_long(
    model_name=MODEL_GLOBAL_TRANSFORMER,
    valid_meta=valid_meta,
    pred_norm=pred_norm,
    invalid_meta=invalid_meta
)

long_parts.append(long_global_transformer)
add_intended_coverage(intended_coverage_rows, MODEL_GLOBAL_TRANSFORMER, ALL_KEYS)

print("Global Transformer long rows:", long_global_transformer.shape)


# ------------------------------------------------------------
# 9.3 Disease Transformer: one Transformer per disease
# ------------------------------------------------------------

for disease in ["ARI", "ILI"]:
    print("\n============================================================")
    print("Training model 3:", MODEL_DISEASE_TRANSFORMER, "| disease:", disease)
    print("============================================================")

    tf.keras.backend.clear_session()

    keys_disease = [key for key in ALL_KEYS if key[0] == disease]

    X_seq_d, Y_d, D_ids_d, L_ids_d = build_supervised_samples(keys_disease)

    print(f"{disease} Transformer training samples:", X_seq_d.shape, Y_d.shape)

    transformer = build_disease_transformer_model(
        num_locations=len(LOCATION_TO_ID)
    )

    hist = fit_model(
        transformer,
        [X_seq_d, L_ids_d],
        Y_d,
        epochs=EPOCHS_DISEASE,
        patience=PATIENCE_DISEASE,
        verbose=1
    )

    training_status_rows.append({
        "model": MODEL_DISEASE_TRANSFORMER,
        "scope": "one_model_per_disease_all_countries",
        "disease": disease,
        "location": "ALL",
        "n_train_samples": len(X_seq_d),
        "epochs_run": len(hist.history["loss"]),
        "final_loss": float(hist.history["loss"][-1]),
        "final_val_loss": float(hist.history["val_loss"][-1]),
        "error": ""
    })

    X_eval_d, D_eval_d, L_eval_d, valid_meta_d, invalid_meta_d = build_eval_inputs(keys_disease)

    pred_norm_d = transformer.predict(
        [X_eval_d, L_eval_d],
        batch_size=256,
        verbose=0
    )

    long_d = predictions_to_long(
        model_name=MODEL_DISEASE_TRANSFORMER,
        valid_meta=valid_meta_d,
        pred_norm=pred_norm_d,
        invalid_meta=invalid_meta_d
    )

    long_parts.append(long_d)
    add_intended_coverage(intended_coverage_rows, MODEL_DISEASE_TRANSFORMER, keys_disease)

    print(f"{disease} Transformer long rows:", long_d.shape)


# ------------------------------------------------------------
# 9.4 Local TCN: one model per disease-country
# ------------------------------------------------------------

for key in ALL_KEYS:
    disease, loc = key

    print("\n============================================================")
    print("Training model 4:", MODEL_LOCAL_TCN, "|", disease, loc)
    print("============================================================")

    tf.keras.backend.clear_session()

    try:
        X_seq_l, Y_l, D_ids_l, L_ids_l = build_supervised_samples([key])

        print(f"{disease} {loc} Local TCN training samples:", X_seq_l.shape, Y_l.shape)

        local_tcn = build_local_tcn_model()

        hist = fit_model(
            local_tcn,
            X_seq_l,
            Y_l,
            epochs=EPOCHS_LOCAL,
            patience=PATIENCE_LOCAL,
            verbose=0
        )

        X_eval_l, D_eval_l, L_eval_l, valid_meta_l, invalid_meta_l = build_eval_inputs([key])

        pred_norm_l = local_tcn.predict(
            X_eval_l,
            batch_size=256,
            verbose=0
        )

        long_l = predictions_to_long(
            model_name=MODEL_LOCAL_TCN,
            valid_meta=valid_meta_l,
            pred_norm=pred_norm_l,
            invalid_meta=invalid_meta_l
        )

        training_status_rows.append({
            "model": MODEL_LOCAL_TCN,
            "scope": "one_model_per_disease_country",
            "disease": disease,
            "location": loc,
            "n_train_samples": len(X_seq_l),
            "epochs_run": len(hist.history["loss"]),
            "final_loss": float(hist.history["loss"][-1]),
            "final_val_loss": float(hist.history["val_loss"][-1]),
            "error": ""
        })

    except Exception as e:
        print("Local TCN failed for", disease, loc, "error:", e)

        X_eval_l, D_eval_l, L_eval_l, valid_meta_l, invalid_meta_l = build_eval_inputs([key])
        pred_norm_l = np.full((len(valid_meta_l), MAX_H), np.nan)

        long_l = predictions_to_long(
            model_name=MODEL_LOCAL_TCN,
            valid_meta=valid_meta_l,
            pred_norm=pred_norm_l,
            invalid_meta=invalid_meta_l
        )

        training_status_rows.append({
            "model": MODEL_LOCAL_TCN,
            "scope": "one_model_per_disease_country",
            "disease": disease,
            "location": loc,
            "n_train_samples": 0,
            "epochs_run": 0,
            "final_loss": np.nan,
            "final_val_loss": np.nan,
            "error": str(e)[:300]
        })

    long_parts.append(long_l)
    add_intended_coverage(intended_coverage_rows, MODEL_LOCAL_TCN, [key])

    print(f"{disease} {loc} Local TCN long rows:", long_l.shape)


# ============================================================
# 10. Combine results, metrics and audit
# ============================================================

dl_long = pd.concat(long_parts, ignore_index=True)
dl_country_h = build_country_h(dl_long)

training_status = pd.DataFrame(training_status_rows)
intended_coverage = (
    pd.DataFrame(intended_coverage_rows)
    .drop_duplicates()
    .reset_index(drop=True)
)

dl_n_audit = build_n_audit(
    country_h=dl_country_h,
    expected_all=expected_all,
    intended_coverage=intended_coverage
)

dl_ari_horizon = build_horizon_summary(dl_country_h, "ARI")
dl_ili_horizon = build_horizon_summary(dl_country_h, "ILI")

audit_agg = (
    dl_n_audit
    .groupby(["model", "disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        scored_points=("scored_points", "sum"),
        missing_due_to_model_or_pred_nan=("missing_due_to_model_or_pred_nan", "sum")
    )
    .sort_values(["model", "disease", "h"])
    .reset_index(drop=True)
)

print("\n============================================================")
print("Final DL long table")
print("============================================================")
print(dl_long.shape)
display(dl_long.head())

print("\n============================================================")
print("Final DL country-horizon table")
print("============================================================")
print(dl_country_h.shape)
display(dl_country_h.head())

print("\n============================================================")
print("N audit aggregated by model / disease / horizon")
print("============================================================")
display(audit_agg)

print("\n============================================================")
print("ARI DL horizon summary")
print("============================================================")
display(dl_ari_horizon)

print("\n============================================================")
print("ILI DL horizon summary")
print("============================================================")
display(dl_ili_horizon)

print("\n============================================================")
print("Training status")
print("============================================================")
display(training_status)


# ============================================================
# 11.  comparability check
# ============================================================

expected_check = (
    audit_agg
    .pivot_table(
        index=["model", "disease"],
        columns="h",
        values="expected_actual_points",
        aggfunc="first"
    )
)

print("\nExpected coverage check:")
display(expected_check)

print("\nExpected correct totals:")
print("ARI: h1=409, h2=401, h3=393, h4=385")
print("ILI: h1=513, h2=503, h3=493, h4=483")

bad_missing = audit_agg[audit_agg["missing_due_to_model_or_pred_nan"] != 0]

if len(bad_missing) == 0:
    print("\nOK: no predictions missing relative to expected actual targets.")
else:
    print("\nWARNING: some model/disease/horizon has missing predictions.")
    display(bad_missing)


# ============================================================
# 12. Save outputs
# ============================================================

dl_long.to_csv(OUT_DIR / "dl_candidates_rolling_2023_long.csv", index=False)
dl_country_h.to_csv(OUT_DIR / "dl_candidates_country_h_2023.csv", index=False)

expected_all.to_csv(OUT_DIR / "dl_candidates_expected_points_2023.csv", index=False)
intended_coverage.to_csv(OUT_DIR / "dl_candidates_intended_coverage_2023.csv", index=False)
dl_n_audit.to_csv(OUT_DIR / "dl_candidates_n_audit_2023.csv", index=False)

dl_ari_horizon.to_csv(OUT_DIR / "dl_candidates_ari_horizon.csv", index=False)
dl_ili_horizon.to_csv(OUT_DIR / "dl_candidates_ili_horizon.csv", index=False)

training_status.to_csv(OUT_DIR / "dl_candidates_training_status.csv", index=False)
audit_agg.to_csv(OUT_DIR / "dl_candidates_n_audit_aggregated.csv", index=False)

zip_path = shutil.make_archive("/content/dl_outputs_fixed", "zip", OUT_DIR)

print("\nSaved outputs to:", OUT_DIR)
print("ZIP:", zip_path)

files.download(zip_path)

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Upload the two raw incidence CSV files: one ARI and one ILI.


Saving latest-ARI_incidence.csv to latest-ARI_incidence.csv
Saving latest-ILI_incidence.csv to latest-ILI_incidence.csv

Uploaded files:
['latest-ARI_incidence.csv', 'latest-ILI_incidence.csv']

ARI shape: (6685, 3)
ILI shape: (10646, 3)


,location,truth_date,value
0,BE,2014-10-05,1407.5
1,BE,2014-10-12,1493.7
2,BE,2014-10-19,1441.3
3,BE,2014-10-26,1514.0
4,BE,2014-11-02,1401.2


,location,truth_date,value
0,AT,2014-10-05,589.6
1,AT,2014-10-12,547.3
2,AT,2014-10-19,768.8
3,AT,2014-10-26,696.1
4,AT,2014-11-02,762.6



train weeks: 469 2014-01-05 00:00:00 -> 2022-12-25 00:00:00
test weeks : 53 2023-01-01 00:00:00 -> 2023-12-31 00:00:00

USING FIXED LOCAL-COMPARABLE COVERAGE
ok_ari: ['BE', 'BG', 'CZ', 'DE', 'EE', 'LT', 'RO', 'SI'] n = 8
ok_ili: ['BE', 'CZ', 'EE', 'GR', 'IE', 'LT', 'NL', 'PL', 'RO', 'SI'] n = 10

Total disease-country series: 18
[('ARI', 'BE'), ('ARI', 'BG'), ('ARI', 'CZ'), ('ARI', 'DE'), ('ARI', 'EE'), ('ARI', 'LT'), ('ARI', 'RO'), ('ARI', 'SI'), ('ILI', 'BE'), ('ILI', 'CZ'), ('ILI', 'EE'), ('ILI', 'GR'), ('ILI', 'IE'), ('ILI', 'LT'), ('ILI', 'NL'), ('ILI', 'PL'), ('ILI', 'RO'), ('ILI', 'SI')]

Training model 1: DL_GlobalGRU_all_infections_all_countries
Global GRU training samples: (5814, 104, 3) (5814, 4)
Epoch 1/45
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.2178 - mae: 0.4941 - val_loss: 0.1220 - val_mae: 0.3182 - learning_rate: 0.0010
Epoch 2/45
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.1495 - mae: 0.3802 - val_loss: 0.1113 - val_mae: 0.2976 - learning_rate: 0.0010


...


Expected coverage check:


h                                                            1    2    3    4
model                                             disease                    
DL_DiseaseTransformer_all_countries               ARI      409  401  393  385
                                                  ILI      513  503  493  483
DL_GlobalGRU_all_infections_all_countries         ARI      409  401  393  385
                                                  ILI      513  503  493  483
DL_GlobalTransformer_all_infections_all_countries ARI      409  401  393  385
                                                  ILI      513  503  493  483
DL_LocalTCN_each_country                          ARI      409  401  393  385
                                                  ILI      513  503  493  483


Expected correct totals:
ARI: h1=409, h2=401, h3=393, h4=385
ILI: h1=513, h2=503, h3=493, h4=483

OK: no predictions missing relative to expected actual targets.

Saved outputs to: /content/dl_outputs_fixed
ZIP: /content/dl_outputs_fixed.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>